In [12]:
import pandas as pd
import numpy as np

# Cargar el dataset (ajusta el nombre del archivo y la ruta si es necesario)
# Si te da error de codificación, a veces los datos del gobierno usan encoding='latin-1'
df = pd.read_csv('../data/raw/Calidad_del_Agua_para_Consumo_Humano_en_Colombia_20260905.csv')

# Mostrar las primeras 5 filas para entender la estructura
df.head()

,DepartamentoCodigo,Departamento,MunicipioCodigo,Municipio,Año,IRCA,Nivel de riesgo,IRCAurbano,Nivel de riesgo urbano,IRCArural,Nivel de riesgo rural
0,11,"Bogotá, D.C.",#TODOS,#TODOS,"2,024",3.6,Sin riesgo,1.1,Sin riesgo,14.8,Riesgo medio
1,13,Bolívar,#TODOS,#TODOS,"2,024",11.3,Bajo riesgo,12.4,Bajo riesgo,6.5,Bajo riesgo
2,15,Boyacá,#TODOS,#TODOS,"2,024",7.5,Bajo riesgo,4.8,Sin riesgo,22,Riesgo medio
3,17,Caldas,#TODOS,#TODOS,"2,024",12.1,Bajo riesgo,0.8,Sin riesgo,49.8,Alto riesgo
4,18,Caquetá,#TODOS,#TODOS,"2,024",6.6,Bajo riesgo,6.6,Bajo riesgo,ND,ND


In [13]:
print("--- NÚMERO DE REGISTROS Y ATRIBUTOS ---")
print(f"Filas (registros): {df.shape[0]}")
print(f"Columnas (atributos): {df.shape[1]}\n")

print("--- NOMBRES DE COLUMNAS Y TIPOS DE DATOS ---")
# Esto te muestra todas las columnas, si son texto (object) o números (int/float)
df.info()

--- NÚMERO DE REGISTROS Y ATRIBUTOS ---
Filas (registros): 19160
Columnas (atributos): 11

--- NOMBRES DE COLUMNAS Y TIPOS DE DATOS ---
<class 'pandas.DataFrame'>
RangeIndex: 19160 entries, 0 to 19159
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DepartamentoCodigo      19160 non-null  int64  
 1   Departamento            19160 non-null  str    
 2   MunicipioCodigo         19160 non-null  str    
 3   Municipio               19160 non-null  str    
 4   Año                     19160 non-null  str    
 5   IRCA                    19160 non-null  float64
 6   Nivel de riesgo         19160 non-null  str    
 7   IRCAurbano              19160 non-null  str    
 8   Nivel de riesgo urbano  19160 non-null  str    
 9   IRCArural               19160 non-null  str    
 10  Nivel de riesgo rural   19160 non-null  str    
dtypes: float64(1), int64(1), str(9)
memory usage: 1.6 MB


In [14]:
print("--- VALORES FALTANTES (NULOS) POR COLUMNA ---")
print(df.isnull().sum())
print("\nPorcentaje de nulos:")
print((df.isnull().sum() / len(df)) * 100)

print("\n--- REGISTROS DUPLICADOS ---")
duplicados = df.duplicated().sum()
print(f"Total de filas exactamente duplicadas: {duplicados}")

--- VALORES FALTANTES (NULOS) POR COLUMNA ---
DepartamentoCodigo        0
Departamento              0
MunicipioCodigo           0
Municipio                 0
Año                       0
IRCA                      0
Nivel de riesgo           0
IRCAurbano                0
Nivel de riesgo urbano    0
IRCArural                 0
Nivel de riesgo rural     0
dtype: int64

Porcentaje de nulos:
DepartamentoCodigo        0.0
Departamento              0.0
MunicipioCodigo           0.0
Municipio                 0.0
Año                       0.0
IRCA                      0.0
Nivel de riesgo           0.0
IRCAurbano                0.0
Nivel de riesgo urbano    0.0
IRCArural                 0.0
Nivel de riesgo rural     0.0
dtype: float64

--- REGISTROS DUPLICADOS ---
Total de filas exactamente duplicadas: 0


In [16]:
print("--- VALORES ÚNICOS Y CARDINALIDAD ---")
# Cuántos valores diferentes hay en cada columna
print(df.nunique())

print("--- COBERTURA GEOGRÁFICA ---")
# Usando exactamente las mayúsculas y minúsculas del archivo fuente
print(f"Total de Departamentos: {df['Departamento'].nunique()}")
print(f"Total de Municipios: {df['Municipio'].nunique()}")

--- VALORES ÚNICOS Y CARDINALIDAD ---
DepartamentoCodigo          33
Departamento                34
MunicipioCodigo           1096
Municipio                 1016
Año                         18
IRCA                       952
Nivel de riesgo              7
IRCAurbano                 967
Nivel de riesgo urbano       8
IRCArural                  996
Nivel de riesgo rural        8
dtype: int64
--- COBERTURA GEOGRÁFICA ---
Total de Departamentos: 34
Total de Municipios: 1016


In [18]:
print("--- COBERTURA DE FECHAS ---")
# Cambia 'AÑO' por el nombre de tu columna de tiempo
print(f"Año inicial: {df['Año'].min()}")
print(f"Año final: {df['Año'].max()}")

print("\n--- ESTADÍSTICAS DESCRIPTIVAS Y RANGOS (IRCA) ---")
# .describe() te da el promedio, mínimo, máximo y cuartiles de las columnas numéricas
print(df.describe())

--- COBERTURA DE FECHAS ---
Año inicial: 2,007
Año final: 2,024

--- ESTADÍSTICAS DESCRIPTIVAS Y RANGOS (IRCA) ---
       DepartamentoCodigo          IRCA
count        19160.000000  19160.000000
mean            37.694833     21.209880
std             26.199080     21.975167
min              5.000000      0.000000
25%             15.000000      3.400000
50%             25.000000     13.100000
75%             66.000000     33.900000
max             99.000000    100.000000


In [20]:
print("--- VALORES SOSPECHOSOS EN EL IRCA ---")
# El IRCA es un porcentaje, no debería ser menor a 0 ni mayor a 100
# Cambia 'IRCA' por el nombre de tu columna
irca_invalido = df[(df['IRCA'] < 0) | (df['IRCA'] > 100)]
print(f"Registros con IRCA fuera de rango (0-100): {len(irca_invalido)}")

# Identificar si hay municipios mal escritos o con caracteres raros
# Mostrar una muestra de departamentos para ver si hay inconsistencias (ej: "BOGOTA" vs "BOGOTÁ")
print("\nLista de departamentos únicos para revisión visual:")
print(df['Departamento'].unique())

--- VALORES SOSPECHOSOS EN EL IRCA ---
Registros con IRCA fuera de rango (0-100): 0

Lista de departamentos únicos para revisión visual:
<StringArray>
[                                            'Bogotá, D.C.',
                                                  'Bolívar',
                                                   'Boyacá',
                                                   'Caldas',
                                                  'Caquetá',
                                                    'Cauca',
                                                    'Cesar',
                                                  'Córdoba',
                                             'Cundinamarca',
                                                    'Chocó',
                                                    'Huila',
                                               'La Guajira',
                                                'Magdalena',
                                                     'Me